# UDCF (LBCF, Ai et al. 2022, WWW'22) aplicado a base Hillstrom - versao ONE-HOT

Codigo C++ original dos autores, sem reimplementacao.

Configuracao desta versao:
- **Hiperparametros**: defaults dos autores, sem nenhum ajuste (`min_node_size=50`, `alpha=0.05`, `imbalance_penalty=0.01`, `mtry=3`, `num_trees=300`)
- **Amostra/predicao**: 100% da base, `predict_oob` (predicao honesta out-of-bag)
- **Codificacao de `zip_code`/`channel`**: ONE-HOT (uma coluna binaria por categoria) -> 11 features no total

Esta versao e identica ao notebook `UDCF_Hillstrom_Colab.ipynb` original; o nome so deixa explicito qual codificacao categorica foi usada, para comparar com a versao ordinal (`UDCF_Hillstrom_Colab_Ordinal.ipynb`).

## 1. Clonar o repositorio dos autores e instalar ferramentas de build

In [ ]:
!git clone -q https://github.com/www2022paper/WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS.git
!apt-get -qq update && apt-get -qq install -y cmake g++

## 2. Extrair o codigo C++ do UDCF (vem zipado dentro do repositorio)

In [ ]:
import zipfile

BASE = "WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/LBCF"
with zipfile.ZipFile(f"{BASE}/LBCF_RCT.zip") as z:
    z.extractall(BASE)

## 3. Carregar a base Hillstrom (dados brutos)

Base publica Hillstrom (MineThatData E-Mail Analytics, 2008): experimento aleatorizado de e-mail marketing com 64.000 clientes, 3 bracos de tratamento (`segment`) e outcome `conversion`.

In [ ]:
import pandas as pd

HILLSTROM_URL = (
    "http://www.minethatdata.com/"
    "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
)
df = pd.read_csv(HILLSTROM_URL)
print("Base Hillstrom original:", df.shape)
df.head()

## 4. Analise exploratoria dos dados (EDA)

Antes de codificar as variaveis e estimar o CATE, descrevemos a base, avaliamos sua qualidade e verificamos se a aleatorizacao do experimento se sustenta empiricamente nas covariaveis observadas -- condicao necessaria para interpretacao causal das estimativas produzidas pelo UDCF.

### 4.1 Dicionario de dados e visao geral

| variavel | tipo | descricao |
|---|---|---|
| `recency` | numerica | meses desde a ultima compra |
| `history` | numerica | valor gasto (US$) nos 12 meses anteriores ao experimento |
| `mens` | binaria | comprou roupa masculina no historico |
| `womens` | binaria | comprou roupa feminina no historico |
| `newbie` | binaria | cliente novo (cadastro nos ultimos 12 meses) |
| `zip_code` | categorica | Urban / Surburban / Rural |
| `channel` | categorica | Phone / Web / Multichannel |
| `segment` | tratamento | No E-Mail (controle) / Mens E-Mail / Womens E-Mail |
| `conversion` | outcome | comprou apos o envio (0/1) -- variavel de interesse deste trabalho |
| `visit`, `spend` | nao utilizadas | pos-tratamento; nao usadas como covariavel (vazamento) |

In [ ]:
print("Tipos de dado:")
print(df.dtypes)
print()
print("Resumo estatistico (covariaveis numericas):")
print(df[["recency", "history", "mens", "womens", "newbie"]].describe().round(2))

### 4.2 Qualidade do dado

In [ ]:
print("Valores faltantes por coluna:")
print(df.isna().sum())
print()
print("Linhas duplicadas:", df.duplicated().sum())

### 4.3 Distribuicoes univariadas

`history` apresenta forte assimetria a direita (cauda longa de clientes de alto gasto); as demais covariaveis tem distribuicao proxima da uniforme entre suas categorias, consistente com um desenho experimental balanceado.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

axes[0, 0].hist(df["recency"], bins=12, color="#4C72B0")
axes[0, 0].set_title("recency")

axes[0, 1].hist(df["history"], bins=40, color="#4C72B0")
axes[0, 1].set_title("history")

for ax, col in zip([axes[0, 2], axes[0, 3]], ["mens", "womens"]):
    df[col].value_counts().sort_index().plot(kind="bar", ax=ax, color="#55A868")
    ax.set_title(col)

for ax, col in zip([axes[1, 0], axes[1, 1], axes[1, 2]], ["newbie", "zip_code", "channel"]):
    df[col].value_counts().sort_index().plot(kind="bar", ax=ax, color="#55A868")
    ax.set_title(col)

axes[1, 3].axis("off")
plt.tight_layout()
plt.show()

### 4.4 Checagem de balanceamento entre grupos (Table 1)

Comparamos a distribuicao de cada covariavel entre os tres bracos do experimento (ANOVA para covariaveis continuas, qui-quadrado para categoricas/binarias). A ausencia de diferencas estatisticamente significativas (p>0.05) corrobora a validade da aleatorizacao, pre-requisito para interpretacao causal das estimativas de CATE produzidas na secao seguinte.

In [ ]:
from scipy import stats

balance_rows = []
for col in ["recency", "history"]:
    groups = [g[col].values for _, g in df.groupby("segment")]
    _, p = stats.f_oneway(*groups)
    balance_rows.append((col, "ANOVA", p))

for col in ["mens", "womens", "newbie", "zip_code", "channel"]:
    tab = pd.crosstab(df[col], df["segment"])
    _, p, _, _ = stats.chi2_contingency(tab)
    balance_rows.append((col, "qui-quadrado", p))

balance = pd.DataFrame(balance_rows, columns=["covariavel", "teste", "p_valor"])
print(balance.round(4).to_string(index=False))

### 4.5 Desfecho: taxa de conversao por grupo (ATE naive)

A media das estimativas de CATE produzidas pelo UDCF, agregada por tratamento, e esperada se aproximar deste ATE naive -- usamos este numero como checagem de calibracao do modelo na secao final.

In [ ]:
rate = df.groupby("segment")["conversion"].agg(["mean", "count"])
print(rate.round(5))

base_rate = rate.loc["No E-Mail", "mean"]
print()
print("Uplift Mens E-Mail vs controle:  ", round(rate.loc["Mens E-Mail", "mean"] - base_rate, 5))
print("Uplift Womens E-Mail vs controle:", round(rate.loc["Womens E-Mail", "mean"] - base_rate, 5))

rate["mean"].plot(kind="bar", color=["#C44E52", "#4C72B0", "#55A868"], figsize=(5, 4))
plt.ylabel("taxa de conversao")
plt.title("Taxa de conversao por grupo de tratamento")
plt.tight_layout()
plt.show()

### 4.6 Cruzamentos bivariados

Taxa de conversao por `zip_code` e por `channel`, cruzada com o grupo de tratamento -- uma primeira intuicao visual de heterogeneidade a ser confirmada (ou nao) pelo CATE estimado.

In [ ]:
for col in ["zip_code", "channel"]:
    print(f"Taxa de conversao por {col} x segment:")
    print(df.groupby([col, "segment"])["conversion"].mean().unstack().round(5))
    print()

## 5. Pre-processamento: codificacao one-hot e montagem do arquivo de entrada

- Outcome (Y): `conversion`
- Tratamento multi-nivel (K=2): `No E-Mail` = controle, `Mens E-Mail` = T1, `Womens E-Mail` = T2
- Features: `recency, history, mens, womens, newbie` + ONE-HOT de `zip_code` (3 colunas) e `channel` (3 colunas) = 11 features

Optamos pela codificacao one-hot (em vez de ordinal) por preservar a natureza nominal de `zip_code` e `channel`, evitando impor uma ordem artificial entre categorias sem hierarquia natural que poderia distorcer os splits da arvore causal.

In [ ]:
feature_cols = ["recency", "history", "mens", "womens", "newbie"]
X = df[feature_cols].astype(float).copy()
zip_dummies = pd.get_dummies(df["zip_code"], prefix="zip", dtype=float)
channel_dummies = pd.get_dummies(df["channel"], prefix="channel", dtype=float)
X = pd.concat([X, zip_dummies, channel_dummies], axis=1)

Y = df["conversion"].astype(float)
code = df["segment"].map({"No E-Mail": 0, "Mens E-Mail": 1, "Womens E-Mail": 2})
T1 = (code == 1).astype(float)
T2 = (code == 2).astype(float)

design = pd.concat([X, Y.rename("Y"), T1.rename("T1"), T2.rename("T2")], axis=1)

n_features = X.shape[1]
outcome_index = n_features
treatment_index = [n_features + 1, n_features + 2]

data_path = f"{BASE}/UDCF_RCT/core/hillstrom_udcf_input.txt"
design.to_csv(data_path, sep=" ", header=False, index=False)

print("numero de features:", n_features)
print("outcome_index:", outcome_index, "| treatment_index:", treatment_index)
print("linhas x colunas do arquivo de entrada:", design.shape)
design.head()

## 6. Gerar o `main.cpp` adaptado

So os indices de coluna e os caminhos de arquivo mudam. Hiperparametros = defaults dos autores (`ForestTestUtilities::default_options`, sem edicao). Predicao = `predict_oob` (100% da amostra, honesta).

In [ ]:
main_cpp = f'''#include <iostream>
#include <string>
#include <unistd.h>

#include "tree/Tree.h"
#include "prediction/DefaultPredictionStrategy.h"
#include "commons/utility.h"
#include "forest/ForestPredictor.h"
#include "forest/ForestTrainer.h"
#include "utilities/FileTestUtilities.h"
#include "utilities/ForestTestUtilities.h"
#include "forest/ForestTrainers.h"
#include "forest/ForestPredictors.h"
#include "analysis/SplitFrequencyComputer.h"
using namespace grf;

void update_predictions_file(const std::string& file_name,
                             const std::vector<Prediction>& predictions) {{
  std::vector<std::vector<double>> values;
  values.reserve(predictions.size());
  for (const auto& prediction : predictions) {{
    values.push_back(prediction.get_predictions());
  }}
  FileTestUtilities::write_csv_file(file_name, values);
  std::cout << "success! predictions dump to " << file_name << std::endl;
}}

int main()
{{
    auto data_vec = load_data("../hillstrom_udcf_input.txt");
    Data data(data_vec);
    data.set_outcome_index({outcome_index});
    data.set_treatment_index({{{treatment_index[0]}, {treatment_index[1]}}});

    size_t num_treatments = 2;

    ForestTrainer trainer = udcf_trainer(num_treatments, 1, true);
    ForestOptions options = ForestTestUtilities::default_options(true, 1);
    Forest forest = trainer.train(data, options);

    std::cout << "FOREST_INFO num_trees=" << forest.get_trees().size()
               << " num_variables=" << forest.get_num_variables() << std::endl;
    const auto& first_tree = forest.get_trees()[0];
    size_t root = first_tree->get_root_node();
    size_t total_nodes = first_tree->get_child_nodes()[0].size();
    bool root_is_leaf = first_tree->is_leaf(root);
    std::cout << "TREE0_INFO total_nodes=" << total_nodes
               << " root=" << root
               << " root_is_leaf=" << (root_is_leaf ? "true" : "false") << std::endl;
    size_t non_leaf_count = 0;
    for (size_t n = 0; n < total_nodes; n++) {{
      if (!first_tree->is_leaf(n)) {{
        non_leaf_count++;
      }}
    }}
    std::cout << "TREE0_INFO non_leaf_node_count=" << non_leaf_count << std::endl;

    SplitFrequencyComputer freq_computer;
    std::vector<std::vector<size_t>> freq = freq_computer.compute(forest, 30);
    std::vector<size_t> total_per_var(forest.get_num_variables(), 0);
    for (const auto& depth_counts : freq) {{
      for (size_t v = 0; v < depth_counts.size(); v++) {{
        total_per_var[v] += depth_counts[v];
      }}
    }}
    std::cout << "SPLIT_FREQ ";
    for (size_t v = 0; v < total_per_var.size(); v++) {{
      std::cout << v << ":" << total_per_var[v] << " ";
    }}
    std::cout << std::endl;

    ForestPredictor predictor = udcf_predictor(1, num_treatments, 1);

    std::vector<Prediction> predictions = predictor.predict_oob(forest, data, false);
    update_predictions_file("../hillstrom_udcf_predictions.txt", predictions);

    return 0;
}}
'''

with open(f"{BASE}/UDCF_RCT/core/main.cpp", "w") as f:
    f.write(main_cpp)

print("main.cpp gerado com sucesso.")

## 7. Compilar (cmake + make, igual ao README original do repositorio)

In [ ]:
%%bash
cd WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/LBCF/UDCF_RCT/core
rm -rf build
mkdir build
cd build
cmake .. -DCMAKE_BUILD_TYPE=Release
make -j4

## 8. Rodar o UDCF treinado (o binario original dos autores)

In [ ]:
%%bash
cd WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/LBCF/UDCF_RCT/core/build
./UDCF_RCT

## 9. Ler o CATE estimado e resumir os resultados

In [ ]:
preds = pd.read_csv(
    f"{BASE}/UDCF_RCT/core/hillstrom_udcf_predictions.txt",
    header=None, sep=r",\s*", engine="python",
)
preds.columns = ["cate_mens_email", "cate_womens_email"]

out = pd.concat([df.reset_index(drop=True), preds], axis=1)
out.to_csv("hillstrom_udcf_cate_onehot.csv", index=False)

print("=== Resumo do CATE estimado (UDCF, codificacao ONE-HOT) ===")
for nome, col in [("Mens E-Mail", "cate_mens_email"), ("Womens E-Mail", "cate_womens_email")]:
    c = out[col]
    print(f"\n{nome}")
    print(f"  media (CATE medio / ATE aproximado): {c.mean():.4f}")
    print(f"  desvio padrao entre usuarios:         {c.std():.4f}")
    print(f"  minimo / maximo:                      {c.min():.4f} / {c.max():.4f}")

naive_t1 = Y[T1 == 1].mean() - Y[(T1 == 0) & (T2 == 0)].mean()
naive_t2 = Y[T2 == 1].mean() - Y[(T1 == 0) & (T2 == 0)].mean()
print("\n=== Diferenca simples de medias (ATE naive, para conferencia) ===")
print(f"  Mens E-Mail vs controle:   {naive_t1:.4f}")
print(f"  Womens E-Mail vs controle: {naive_t2:.4f}")

print("\nArquivo salvo: hillstrom_udcf_cate_onehot.csv (inclui CATE por usuario)")